In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from azure.storage.blob import BlobServiceClient
from tqdm import tqdm

# Configuration
STORAGE_ACCOUNT = "solarflarestorageproject"
STORAGE_KEY = os.environ.get("AZURE_STORAGE_KEY", "")
RAW_CONTAINER = "raw"
PROC_CONTAINER = "processed"
IMAGE_SIZE = (224, 224)

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [2]:
# Run this cell manually each session - do not save with key filled in
import os
os.environ["AZURE_STORAGE_KEY"] = ""
STORAGE_KEY = os.environ.get("AZURE_STORAGE_KEY", "")
print("Key set for this session.")

Key set for this session.


In [3]:
# Connect to both containers
raw_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
).get_container_client(RAW_CONTAINER)

proc_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
).get_container_client(PROC_CONTAINER)

blobs = list(raw_client.list_blobs())
print(f"Connected successfully.")
print(f"Total images in raw container: {len(blobs)}")

Connected successfully.
Total images in raw container: 0


In [4]:
def parse_label(blob_name):
    """Extract flare class from filename (X, M, C, B, or N for no flare)."""
    filename = blob_name.split("/")[-1]
    for cls in ["X", "M", "C", "B"]:
        if filename.startswith(cls):
            return cls
    return "N"

def validate_image(img, filename):
    """Run quality checks on a single image."""
    arr = np.array(img)
    return {
        "filename":   filename,
        "label":      parse_label(filename),
        "width":      img.width,
        "height":     img.height,
        "mode":       img.mode,
        "is_blank":   bool(arr.std() < 1e-5),
        "min_pixel":  float(arr.min()),
        "max_pixel":  float(arr.max()),
        "mean_pixel": float(arr.mean()),
        "std_pixel":  float(arr.std()),
        "valid":      img.size == IMAGE_SIZE and arr.std() >= 1e-5
    }

def normalize_image(img):
    """Normalize pixel values to 0-1 range."""
    arr = np.array(img).astype(np.float32)
    arr_min, arr_max = arr.min(), arr.max()
    if arr_max - arr_min > 0:
        arr = (arr - arr_min) / (arr_max - arr_min)
    return arr

print("Functions defined successfully.")

Functions defined successfully.


In [7]:
def run_etl(raw_client, proc_client, max_images=None):
    """
    Main ETL pipeline:
    1. Download images from raw container
    2. Validate each image
    3. Normalize pixel values
    4. Upload cleaned images to processed container
    5. Return validation report
    """
    blobs = list(raw_client.list_blobs())
    if max_images:
        blobs = blobs[:max_images]

    print(f"Processing {len(blobs)} images...")
    report = []

    for blob in tqdm(blobs):
        try:
            data = raw_client.download_blob(blob.name).readall()
            img = Image.open(BytesIO(data)).convert("L")
            stats = validate_image(img, blob.name)
            report.append(stats)

            if not stats["valid"]:
                continue

            arr_norm = normalize_image(img)
            img_norm = Image.fromarray((arr_norm * 255).astype(np.uint8))
            buf = BytesIO()
            img_norm.save(buf, format="PNG")
            buf.seek(0)

            clean_name = blob.name.replace("magnetograms/", "clean/")
            proc_client.upload_blob(name=clean_name, data=buf, overwrite=True)

        except Exception as e:
            print(f"Error processing {blob.name}: {e}")
            continue

    return pd.DataFrame(report)

print("ETL pipeline function defined successfully.")

ETL pipeline function defined successfully.


In [8]:
# Execute ETL pipeline
report_df = run_etl(raw_client, proc_client)

if len(report_df) > 0:
    os.makedirs("outputs", exist_ok=True)
    report_df.to_csv("outputs/validation_report.csv", index=False)

    print("ETL Summary:")
    print(f"Total images processed:  {len(report_df)}")
    print(f"Valid images:            {report_df['valid'].sum()}")
    print(f"Invalid/blank images:    {(~report_df['valid']).sum()}")
    print(f"\nLabel distribution:")
    print(report_df['label'].value_counts())
else:
    print("No images found in raw container yet.")
    print("Re-run this cell once the dataset has been uploaded.")